In [5]:
import pandas as pd
import logging
from datetime import datetime


In [6]:

# Set up logging configuration

logging.basicConfig(  
    filename = "cleaning.log",
    level = logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s  "
) # write logs to cleaning.log file, set logging level to INFO, and specify the log message format


start_time = datetime.now() # record the start time of the cleaning process

# Load Data Set

try:
    df = pd.read_csv("messy_data.csv") # read the messy data from a CSV file into a pandas DataFrame
    logging.info("Data loaded successfully.") # log a message indicating successful data loading

except Exception as e:
    logging.error("Error occurred while loading data: %s", str(e)) # log an error message if there is an exception while loading the data
    print("Error :", e)
    exit() 


In [7]:

# store initial information

rows_before = len(df)

duplicates_removed = 0
missing_values_removed = 0
invalid_values_removed = 0

#Remove Dulpicates

duplicates_removed = df.duplicated().sum() 

df = df.drop_duplicates()
logging.info(f"Duplicates removed : {duplicates_removed}") 

#remove extra spaces from string columns

text_columns = ["Name", "City"]

for col in text_columns:
    df[col] = df[col].str.strip()

logging.info("Extra spaces removed from string columns.")


In [8]:

# Standardize Tex

df["City"] = df["City"].str.title()

logging.info("City names standardized.")
 
# Handle Missing Name 

missing_name = df["Name"].isnull().sum()

df["Name"] = df["Name"].replace("nan", "Unknown")
df["Name"] = df["Name"].fillna("Unknown")
 
# Handle Missing City
 
missing_city = df["City"].isnull().sum()

df["City"] = df["City"].replace("nan", "Unknown")
df["City"] = df["City"].fillna("Unknown")
 
# Handle Missing Age 

age_mean = df["Age"].mean()

missing_age = df["Age"].isnull().sum()

df["Age"] = df["Age"].fillna(age_mean)
 
# Handle Missing Marks 

marks_mean = df["Marks"].mean()

missing_marks = df["Marks"].isnull().sum()

df["Marks"] = df["Marks"].fillna(marks_mean)

missing_values_filled = (
    missing_name +
    missing_city +
    missing_age +
    missing_marks
)

logging.info(f"Missing values filled: {missing_values_filled}")
 


In [9]:
# Remove Invalid Age 

before = len(df)

df = df[df["Age"] >= 0]

after = len(df)

invalid_rows_removed = before - after
 
# Remove Invalid Marks 

before = len(df)

df = df[
    (df["Marks"] >= 0) &
    (df["Marks"] <= 100)
]

after = len(df)

invalid_rows_removed += before - after

logging.info(f"Invalid rows removed: {invalid_rows_removed}")
 
# Convert Data Types 

df["Age"] = df["Age"].astype(int)
df["Marks"] = df["Marks"].round(2)


In [10]:
 
# Save Cleaned Dataset 
df.to_csv(
    "cleaned_data.csv",
    index=False
)

logging.info("Cleaned dataset saved.")
 
# Statistics 

rows_after = len(df)

execution_time = datetime.now() - start_time
 


In [11]:
# Generate Report 

report = f"""
=========================================
DATA CLEANING REPORT
=========================================

Rows Before Cleaning : {rows_before}

Rows After Cleaning  : {rows_after}

Duplicates Removed   : {duplicates_removed}

Missing Values Fixed : {missing_values_filled}

Invalid Rows Removed : {invalid_rows_removed}

Average Age          : {df['Age'].mean():.2f}

Average Marks        : {df['Marks'].mean():.2f}

Execution Time       : {execution_time}

Output File          : cleaned_data.csv

=========================================
"""

with open("cleaning_report.txt", "w") as file:
    file.write(report)

logging.info("Cleaning report generated.")

print(report)

print("Cleaning Completed Successfully!")


DATA CLEANING REPORT

Rows Before Cleaning : 10

Rows After Cleaning  : 7

Duplicates Removed   : 1

Missing Values Fixed : 4

Invalid Rows Removed : 2

Average Age          : 20.14

Average Marks        : 83.46

Execution Time       : 0:00:00.076691

Output File          : cleaned_data.csv


Cleaning Completed Successfully!
